In [0]:
import pyspark.sql.functions as F

In [0]:
trans_df= spark.read.table("payment_app.silver.transactions")
cbs_df=spark.read.table("payment_app.silver.cbs")
settle_df=spark.read.table("payment_app.silver.settlement")

In [0]:
COU_df = trans_df.join(cbs_df, trans_df.txn_id == cbs_df.txn_id) \
                 .select(trans_df.txn_id, trans_df.biller, trans_df.amount, trans_df.status, trans_df.transaction_source, trans_df.transaction_date)

In [0]:
COU_success= COU_df.filter((F.col("CBS_status")== 'Success') & (F.col("status")== 'Success'))
                

In [0]:
COU_exceptions = COU_df.filter(F.col("CBS_status") != F.col("status"))

In [0]:
BOU_df = trans_df.join(settle_df, trans_df.txn_id == settle_df.txn_id) \
                 .select(trans_df.txn_id, trans_df.biller, trans_df.amount, trans_df.status, trans_df.transaction_source, trans_df.transaction_date)

In [0]:
BOU_success= BOU_df.filter((F.col("settle_status")== 'Success') & (F.col("status")== 'Success'))

In [0]:
BOU_exceptions = BOU_df.filter(F.col("settle_status") != F.col("status"))

In [0]:
success_df = COU_success.union(BOU_success)
exceptions_df = COU_exceptions.union(BOU_exceptions)
display(success_df)
display(exceptions_df)
#Write Success and Exceptions to Gold
success_df.write.mode("overwrite").saveAsTable("payment_app.gold.success")
exceptions_df.write.mode("overwrite").saveAsTable("payment_app.gold.exceptions")

In [0]:
df = spark.read.table("payment_app.silver.settlement")
df.printSchema()